In [1]:
import sys

print(sys.executable)

/home/yuli6752/miniconda3/envs/chinese_nlp/bin/python


In [2]:
import pandas as pd
import numpy as np
import sklearn
import jieba
import gensim
import networkx

print("Environment check passed!")

Environment check passed!


In [3]:
import torch
import transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

PyTorch: 2.13.0+cu130
Transformers: 5.15.0


In [4]:
#loading dataset
from datasets import load_dataset

print("datasets imported successfully")

datasets imported successfully


In [5]:
import os

print("Current working directory:")
print(os.getcwd())

Current working directory:
/gorilla/home/yuli6752/chinese-topic-analysis/notebook


In [6]:
import os

path = "../data/raw/train-00000-of-00001.parquet"

print(os.path.exists(path))
print(os.path.getsize(path) / 1024 / 1024)

True
9.958471298217773


In [7]:
#loading streaming
import pandas as pd
import time

path = "../data/raw/train-00000-of-00001.parquet"

start = time.time()

df = pd.read_parquet(path)

print("Loaded!")
print("Shape:", df.shape)
print("Time:", round(time.time() - start, 2), "seconds")

Loaded!
Shape: (6000, 7)
Time: 0.03 seconds


In [8]:
print(df.columns.tolist())

['image', 'title', 'content', 'tag', 'author', 'timestamp', 'link']


In [9]:
df.head(3)

,image,title,content,tag,author,timestamp,link
0,https://image.cache.storm.mg/styles/smg-150x15...,走斑馬線還是被撞！新北高中生怒轟「你他X眼瞎啊」影片瘋傳 駕駛當下行為也曝光,開車不看前方，駕駛被國中生罵爆！新北市日前發生一起擦撞意外，一名國中生踩著斑馬線過馬路時，竟...,風生活,古靜兒,2024-03-03 12:48,https://www.storm.mg/lifestyle/5039178
1,https://image.cache.storm.mg/styles/smg-150x15...,不接受「金門撞船案」專案報告名稱 民進黨團明甲動：力挺國家執法,立委高金素梅日前針對陸船翻覆事件於內政部委員會排定「金門撞船案」專案報告，遭海委會主委管碧玲...,政治,許詠晴,2024-03-03 12:30,https://www.storm.mg/article/5039155
2,https://image.cache.storm.mg/styles/smg-150x15...,開刀住院3天申請實支實付！他見「3項目沒理賠」氣炸，網一面倒不挺,不少民眾都有購買保險，避免在生病住院、意外等情況下支付巨額費用。近日，有網友抱怨，他出院後向...,風生活,王若桐,2024-03-03 12:22,https://www.storm.mg/lifestyle/5039117


In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   image      6000 non-null   str  
 1   title      6000 non-null   str  
 2   content    6000 non-null   str  
 3   tag        6000 non-null   str  
 4   author     6000 non-null   str  
 5   timestamp  6000 non-null   str  
 6   link       6000 non-null   str  
dtypes: str(7)
memory usage: 15.1 MB


In [11]:
df.isnull().sum()

image        0
title        0
content      0
tag          0
author       0
timestamp    0
link         0
dtype: int64

In [12]:
df["tag"].value_counts()

tag
風生活     3034
政治      1364
評論       725
國際       548
國內       140
財經       113
品味生活      52
公共政策       9
中港澳        9
公民運動       3
科技         1
中央社        1
調查         1
Name: count, dtype: int64

In [13]:
print("Number of categories:", df["tag"].nunique())

Number of categories: 13


In [14]:
#establish NLP text
df["text"] = (
    df["title"].fillna("") + " " +
    df["content"].fillna("")
)

print(df[["title", "text"]].head(2))

                                    title  \
0  走斑馬線還是被撞！新北高中生怒轟「你他X眼瞎啊」影片瘋傳　駕駛當下行為也曝光   
1         不接受「金門撞船案」專案報告名稱　民進黨團明甲動：力挺國家執法   

                                                text  
0  走斑馬線還是被撞！新北高中生怒轟「你他X眼瞎啊」影片瘋傳　駕駛當下行為也曝光 開車不看前方，...  
1  不接受「金門撞船案」專案報告名稱　民進黨團明甲動：力挺國家執法 立委高金素梅日前針對陸船翻覆...  


In [15]:
df["text_length"] = df["text"].str.len()

df["text_length"].describe()

count    6000.000000
mean      831.462667
std       363.081542
min        11.000000
25%       584.000000
50%       831.000000
75%      1076.250000
max      4783.000000
Name: text_length, dtype: float64

In [16]:
df[["title", "text", "text_length"]].sort_values(
    "text_length"
).head(10)

,title,text,text_length
3806,閻紀宇專欄：列寧百年,閻紀宇專欄：列寧百年,11
4430,顏厥安專欄：另類選擇的抉擇,顏厥安專欄：另類選擇的抉擇,14
2246,閻紀宇專欄：全世界最酷的獨裁者,閻紀宇專欄：全世界最酷的獨裁者,16
5625,閻紀宇專欄：香蕉共和國的「內戰」危機,閻紀宇專欄：香蕉共和國的「內戰」危機,19
1801,觀點投書：無知竟成解救世界的唯一良藥？,觀點投書：無知竟成解救世界的唯一良藥？,20
116,觀點投書：蘇起秘書長新書觀點何錯之有？,觀點投書：蘇起秘書長新書觀點何錯之有？,20
4425,沈旭暉專欄：索馬利蘭獨立終於被承認在望,沈旭暉專欄：索馬利蘭獨立終於被承認在望,20
4607,吳鯤鵬專欄：王家衛的《繁花》─大夢未醒,吳鯤鵬專欄：王家衛的《繁花》─大夢未醒,20
2535,人物》劍指黨主席？戰鬥藍老大趙少康想好了,人物》劍指黨主席？戰鬥藍老大趙少康想好了,21
4302,朱淑娟專欄：看守政府適合續審四接環評嗎？,朱淑娟專欄：看守政府適合續審四接環評嗎？,21


In [18]:
df["title_length"] = df["title"].str.len()
df["content_length"] = df["content"].str.len()

df[["title", "title_length", "content_length"]].sort_values(
    "content_length"
).head(20)

,title,title_length,content_length
4151,幕後》國造潛艦「瘦馬效應」被笑 海鯤號將由他們操刀大整容,28,0
102,幕後》雲林張家洗臉「不敗女王」！柯文哲要參一腳 賴清德掛帥旗打補選復仇之戰,38,0
5512,黃信維觀點：台灣的難題！「神格」總統賴清德 卻入「少數政府」修羅場,33,0
5513,戴祺修觀點：「侯侯做歹誌」破功─凱旋苑成壓垮侯友宜最後一根稻草,31,0
5515,晏明強觀點：國民黨敗戰啟動接班！侯友宜「謝謝收看」「不敗女王」盧秀燕來了,36,0
5518,蔡宜彣觀點：差點聰明反被聰明誤！柯文哲抓牢年輕人，翻雲覆雨新國會,32,0
5519,人物》賴清德救不了高嘉瑜！港湖女神失去自己 選民終究不以為然,30,0
3421,台美「情境式戀愛關係」無法長久 《彭博》專欄作家談重新評估戰略模糊,33,0
2291,春節前「維修」人數大增？全民瘋醫美少女針藏水貨 畸形恐懼嚴重恐致死,33,0
2293,幕後》「咱們注意保存為要」！調查局特藏文件 有習近平父親是否婚內出軌的秘密,37,0


In [19]:
(df["content_length"] == 0).sum()
(df["content_length"] < 50).sum()

print(
    "Content < 50 chars:",
    (df["content_length"] < 50).sum(),
    "/",
    len(df),
    "=",
    round((df["content_length"] < 50).mean() * 100, 2),
    "%"
)

Content < 50 chars: 236 / 6000 = 3.93 %


In [20]:
df[df["content_length"] < 50]["tag"].value_counts()

tag
政治      107
國際       57
評論       47
國內       14
公共政策      7
財經        2
調查        1
中港澳       1
Name: count, dtype: int64

In [27]:
print("Total articles:", len(df))
print("Articles with empty content:", (df["content_length"] == 0).sum())
print(
    "Percentage:",
    round((df["content_length"] == 0).mean() * 100, 2),
    "%"
)

df[df["content_length"] == 0]["tag"].value_counts()

Total articles: 6000
Articles with empty content: 236
Percentage: 3.93 %


tag
政治      107
國際       57
評論       47
國內       14
公共政策      7
財經        2
調查        1
中港澳       1
Name: count, dtype: int64

In [28]:
#create a cleaned dataset
df_clean = df[df["content_length"] > 0].copy()

print("Original articles:", len(df))
print("Clean articles:", len(df_clean))
print("Removed articles:", len(df) - len(df_clean))

Original articles: 6000
Clean articles: 5764
Removed articles: 236


In [29]:
df_clean["tag"].value_counts()

tag
風生活     3034
政治      1257
評論       678
國際       491
國內       126
財經       111
品味生活      52
中港澳        8
公民運動       3
公共政策       2
科技         1
中央社        1
Name: count, dtype: int64

In [30]:
print("Original tag distribution:")
print(df["tag"].value_counts(normalize=True).round(3))

print("\nCleaned tag distribution:")
print(df_clean["tag"].value_counts(normalize=True).round(3))

Original tag distribution:
tag
風生活     0.506
政治      0.227
評論      0.121
國際      0.091
國內      0.023
財經      0.019
品味生活    0.009
公共政策    0.002
中港澳     0.002
公民運動    0.000
科技      0.000
中央社     0.000
調查      0.000
Name: proportion, dtype: float64

Cleaned tag distribution:
tag
風生活     0.526
政治      0.218
評論      0.118
國際      0.085
國內      0.022
財經      0.019
品味生活    0.009
中港澳     0.001
公民運動    0.001
公共政策    0.000
科技      0.000
中央社     0.000
Name: proportion, dtype: float64


In [31]:
import os

os.makedirs("../data/processed", exist_ok=True)

df_clean.to_parquet(
    "../data/processed/news_clean.parquet",
    index=False
)

print("Saved cleaned dataset!")

Saved cleaned dataset!


In [32]:
df_clean = df[df["content_length"] > 0].copy()

print(df_clean.shape)

df_clean.to_parquet(
    "../data/processed/news_clean.parquet",
    index=False
)

(5764, 11)
